# Data Merger
Merges `steam-200k.csv` (play data) with `games_in_detail_version_clean.csv` (game metadata) into a single `output/steam_merged.csv` for the recommender system.

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import re
import os

os.makedirs('output', exist_ok=True)
print('Libraries loaded.')

Libraries loaded.


## 2. Load & Prepare Play Data

In [2]:
ratings = pd.read_csv('input/steam-200k.csv', encoding='utf-8', encoding_errors='replace')
print(f'Raw rows: {len(ratings):,}')
print(ratings.head())

Raw rows: 200,000
     user_id                        game  behavior  value  extra
0  151603712  The Elder Scrolls V Skyrim  purchase    1.0      0
1  151603712  The Elder Scrolls V Skyrim      play  273.0      0
2  151603712                   Fallout 4  purchase    1.0      0
3  151603712                   Fallout 4      play   87.0      0
4  151603712                       Spore  purchase    1.0      0


In [3]:
# Keep only 'play' rows
play_df = ratings[ratings['behavior'] == 'play'].copy()
play_df.rename(columns={'value': 'hours_played'}, inplace=True)
play_df['hours_played'] = pd.to_numeric(play_df['hours_played'], errors='coerce')
play_df.dropna(subset=['hours_played'], inplace=True)

print(f'Play rows: {len(play_df):,}')
print(f'Unique users: {play_df["user_id"].nunique():,}')
print(f'Unique games: {play_df["game"].nunique():,}')

Play rows: 70,489
Unique users: 11,350
Unique games: 3,600


## 3. Hours → Rating (1–5 scale)

In [4]:
def hours_to_rating(hours):
    if hours == 0:    return 1
    elif hours < 1:   return 1
    elif hours < 5:   return 2
    elif hours < 20:  return 3
    elif hours < 100: return 4
    else:             return 5

play_df['rating'] = play_df['hours_played'].apply(hours_to_rating)

print('Rating distribution:')
print(play_df['rating'].value_counts().sort_index())

Rating distribution:
rating
1    16792
2    19721
3    16793
4    11402
5     5781
Name: count, dtype: int64


## 4. Filter Sparse Users (min 3 games played)

In [5]:
user_counts = play_df['user_id'].value_counts()
play_df = play_df[play_df['user_id'].isin(user_counts[user_counts >= 3].index)]

print(f'Rows after filtering sparse users: {len(play_df):,}')
print(f'Unique users remaining: {play_df["user_id"].nunique():,}')

Rows after filtering sparse users: 61,280
Unique users remaining: 3,466


## 5. Load Games Metadata

In [6]:
games = pd.read_csv('input/games_in_detail_version_clean.csv',
                    encoding='utf-8', encoding_errors='replace')
print(f'Games rows: {len(games):,}')
print(f'Columns: {list(games.columns)}')

Games rows: 122,611
Columns: ['app_id', 'game_title', 'release_date', 'price', 'description', 'cover_image_url', 'positive_reviews', 'negative_reviews', 'avg_playtime_mins', 'developers', 'publishers', 'categories', 'genres', 'tags', 'review_score_pct', 'total_reviews']


## 6. Normalise Game Names for Matching

In [7]:
def normalise(name):
    if pd.isna(name): return ''
    name = str(name).lower().strip()
    name = re.sub(r'[®™©:]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

play_df['game_norm'] = play_df['game'].apply(normalise)
games['title_norm']  = games['game_title'].apply(normalise)

# Deduplicate games by title_norm — keep the row with the most total_reviews.
# Duplicate title_norm entries cause one play row to fan out into multiple
# merged rows, inflating the dataset with phantom duplicates.
before = len(games)
games = games.sort_values('total_reviews', ascending=False)
games = games.drop_duplicates(subset='title_norm', keep='first').copy()
after = len(games)
print(f'Games deduped: {before:,} → {after:,} rows ({before - after:,} duplicates removed)')

# ── Manual map: game_norm (steam-200k) → title_norm (games metadata) ─────────
# These are high-impact games that can't be matched by exact or fuzzy matching
# because their names differ too much or don't exist under that name in the CSV.
# Values are verified against the actual title_norm values in the games CSV.
known_title_norms = set(games['title_norm'].tolist())

MANUAL_MAP = {
    # CoD multiplayer variants → base game (same metadata)
    'call of duty modern warfare 2 - multiplayer':  'call of duty modern warfare 2',
    'call of duty modern warfare 3 - multiplayer':  'call of duty modern warfare 3',
    'call of duty black ops - multiplayer':         'call of duty black ops',
    'call of duty black ops ii - multiplayer':      'call of duty black ops ii',
    'call of duty black ops iii - multiplayer':     'call of duty black ops iii',
}

# Only keep mappings whose target actually exists in the games CSV
MANUAL_MAP = {k: v for k, v in MANUAL_MAP.items() if v in known_title_norms}

play_df['game_norm'] = play_df['game_norm'].map(
    lambda x: MANUAL_MAP.get(x, x)
)

print(f'Manual map entries applied: {len(MANUAL_MAP)}')
print('Sample normalised play names:')
print(play_df[['game', 'game_norm']].drop_duplicates().head(10))

Games deduped: 122,611 → 121,074 rows (1,537 duplicates removed)
Manual map entries applied: 3
Sample normalised play names:
                          game                   game_norm
1   The Elder Scrolls V Skyrim  the elder scrolls v skyrim
3                    Fallout 4                   fallout 4
5                        Spore                       spore
7            Fallout New Vegas           fallout new vegas
9                Left 4 Dead 2               left 4 dead 2
11                    HuniePop                    huniepop
13               Path of Exile               path of exile
15                 Poly Bridge                 poly bridge
17                 Left 4 Dead                 left 4 dead
19             Team Fortress 2             team fortress 2


## 7. Exact Merge

In [8]:
merged = pd.merge(play_df, games, left_on='game_norm', right_on='title_norm', how='left')
# Keep game_norm alive — fuzzy cell needs it; title_norm is redundant after merge
merged.drop(columns=['title_norm', 'behavior', 'extra'],
            inplace=True, errors='ignore')

matched    = merged['game_title'].notna().sum()
total      = len(merged)
match_rate = matched / total * 100

print(f'Total rows    : {total:,}')
print(f'Matched rows  : {matched:,}')
print(f'Match rate    : {match_rate:.1f}%')

Total rows    : 61,280
Matched rows  : 42,288
Match rate    : 69.0%


## 8. Manual Mapping + Fuzzy Matching

In [9]:
if match_rate < 80:
    print('Match rate below 80% — running fuzzy matching with rapidfuzz...')
    from rapidfuzz import process, fuzz

    # Roman numerals + digits guard — prevents sequel mis-matches
    ROMAN = {'i','ii','iii','iv','v','vi','vii','viii','ix','x',
             'xi','xii','xiii','xiv','xv','xvi','xvii','xviii','xix','xx'}

    # Edition/variant words — if one name has these and the other doesn't, reject.
    EDITION_WORDS = {
        'redux', 'remastered', 'remaster', 'legacy', 'exalt', 'ultimate',
        'goty', 'collection', 'complete', 'deluxe', 'enhanced', 'definitive',
        'anniversary', 'directors', 'extended', 'gold', 'platinum', 'premium',
        'classic', 'shooter', 'zombies', 'vietnam', 'hd'
    }

    def extract_numbers(name):
        tokens = re.split(r'[\s\-]+', name)
        nums = set()
        for t in tokens:
            # Catches: plain digits (2, 3), roman numerals (ii, iv), version tokens (v2, v3)
            if t in ROMAN or re.fullmatch(r'\d+', t) or re.fullmatch(r'v\d+', t):
                nums.add(t)
        return nums

    def edition_tokens(name):
        tokens = set(re.split(r'[\s\-]+', name))
        return EDITION_WORDS & tokens

    def word_set(name):
        return frozenset(re.split(r'[\s\-]+', name))

    game_titles    = games['title_norm'].tolist()
    unmatched_mask = merged['game_title'].isna()
    unmatched_names = merged.loc[unmatched_mask, 'game_norm'].unique()

    print(f'Unmatched unique games: {len(unmatched_names)}')

    fuzzy_map = {}
    for name in unmatched_names:
        result = process.extractOne(name, game_titles, scorer=fuzz.WRatio,
                                    score_cutoff=92)
        if result:
            candidate = result[0]
            # Guard 1: number/numeral/version tokens must match
            if extract_numbers(name) != extract_numbers(candidate):
                continue
            # Guard 2: edition/variant words must match
            if edition_tokens(name) != edition_tokens(candidate):
                continue
            # Guard 3: same word-set in different order = different game
            if word_set(name) == word_set(candidate) and name != candidate:
                continue
            # Guard 4: length ratio — rejects garbage like 'fractured space' → 'space space'
            len_ratio = len(name) / max(len(candidate), 1)
            if len_ratio < 0.75 or len_ratio > 1.33:
                continue
            fuzzy_map[name] = candidate

    print(f'Fuzzy matches found: {len(fuzzy_map)}')
    print('Sample fuzzy matches:')
    for k, v in list(fuzzy_map.items())[:15]:
        print(f'  "{k}" → "{v}"')

    # Re-merge unmatched rows using the fuzzy map
    matched_df   = merged[~unmatched_mask].copy()
    unmatched_df = merged[unmatched_mask].copy()

    unmatched_df['fuzzy_norm'] = unmatched_df['game_norm'].map(fuzzy_map)
    unmatched_df.drop(columns=[c for c in games.columns if c in unmatched_df.columns],
                      inplace=True, errors='ignore')

    fuzzy_merged = pd.merge(unmatched_df, games,
                            left_on='fuzzy_norm', right_on='title_norm', how='left')
    fuzzy_merged.drop(columns=['fuzzy_norm', 'title_norm'], inplace=True, errors='ignore')

    merged = pd.concat([matched_df, fuzzy_merged], ignore_index=True)

    matched    = merged['game_title'].notna().sum()
    match_rate = matched / len(merged) * 100
    print(f'Match rate after fuzzy: {match_rate:.1f}%')
else:
    print(f'Match rate is {match_rate:.1f}% — fuzzy matching not needed.')

# Drop game_norm now that matching is complete
merged.drop(columns=['game_norm'], inplace=True, errors='ignore')

Match rate below 80% — running fuzzy matching with rapidfuzz...
Unmatched unique games: 1165
Fuzzy matches found: 130
Sample fuzzy matches:
  "assassin's creed iii" → "assassin’s creed iii"
  "secrets of rtikon" → "secrets of rætikon"
  "diggeronline" → "digger online"
  "assassin's creed brotherhood" → "assassin’s creed brotherhood"
  "kingdoms of amalur reckoning" → "kingdoms of amalur re-reckoning"
  "trackmania stadium" → "trackmania² stadium"
  "don't starve together beta" → "don't starve together"
  "command and conquer 4 tiberian twilight" → "command & conquer 4 tiberian twilight"
  "brtal legend" → "brutal legend"
  "star wars empire at war gold" → "star wars empire at war - gold pack"
  "guns and robots" → "guns and bots"
  "command and conquer red alert 3" → "command & conquer red alert 3"
  "defiance" → "dfiance"
  "knights of pen and paper +1" → "knights of pen and paper +1 edition"
  "relic hunters zero" → "relic hunters zero remix"
Match rate after fuzzy: 71.4%


## 9. Inspect Top Unmatched Games

In [10]:
unmatched = merged[merged['game_title'].isna()]
top_unmatched = (
    unmatched.groupby('game')['user_id']
    .count()
    .sort_values(ascending=False)
    .head(20)
)
print('Top 20 unmatched games (by number of users):')
print(top_unmatched.to_string())

Top 20 unmatched games (by number of users):
game
Counter-Strike Global Offensive                1102
Robocraft                                       350
Heroes & Generals                               300
Call of Duty Modern Warfare 2 - Multiplayer     242
Grand Theft Auto V                              239
Call of Duty Modern Warfare 2                   238
Grand Theft Auto IV                             222
Tomb Raider                                     195
Call of Duty Modern Warfare 3                   179
Dead Island                                     173
Age of Empires II HD Edition                    172
Call of Duty Modern Warfare 3 - Multiplayer     169
Sniper Elite V2                                 169
Metro 2033                                      167
Loadout                                         164
Empire Total War                                161
Nosgoth                                         159
Warface                                         146
Counter-Strike

## 10. Final Dataset Overview

In [11]:
print(f'Final merged shape : {merged.shape}')
print(f'Columns            : {list(merged.columns)}')
print()
print(merged.head())

Final merged shape : (61280, 20)
Columns            : ['user_id', 'game', 'hours_played', 'rating', 'app_id', 'game_title', 'release_date', 'price', 'description', 'cover_image_url', 'positive_reviews', 'negative_reviews', 'avg_playtime_mins', 'developers', 'publishers', 'categories', 'genres', 'tags', 'review_score_pct', 'total_reviews']

     user_id                        game  hours_played  rating    app_id  \
0  151603712  The Elder Scrolls V Skyrim         273.0       5   72850.0   
1  151603712                   Fallout 4          87.0       4  377160.0   
2  151603712                       Spore          14.9       3   17390.0   
3  151603712           Fallout New Vegas          12.1       3   22380.0   
4  151603712               Left 4 Dead 2           8.9       3     550.0   

                    game_title  release_date  price  \
0  The Elder Scrolls V: Skyrim  Nov 10, 2011  19.99   
1                    Fallout 4   Nov 9, 2015   7.99   
2                       SPORE™  Dec 

## 11. Save Output

In [12]:
merged.to_csv('output/steam_merged.csv', index=False, encoding='utf-8')
print('Saved → output/steam_merged.csv')

Saved → output/steam_merged.csv
